In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
import pickle as pkl
from load_data import DataLoader
import numpy as np
# from utils import convert_relation
from expand_subgraph import ExpandSubgraph
import torch
import random

In [4]:
with open("knowledge_graph/KG_data/FB15k-237-betae/id2ent.pkl", "rb") as f:
    id2ent = pkl.load(f)
with open("knowledge_graph/KG_data/FB15k-237-betae/id2rel.pkl", "rb") as f:
    id2rel = pkl.load(f)
with open("knowledge_graph/KG_data/FB15k-237-betae/ent2id.pkl", "rb") as f:
    ent2id = pkl.load(f)

with open("knowledge_graph/queries/train_all_id.pkl", "rb") as f:
    queries = pkl.load(f)

with open("knowledge_graph/KG_data/FB15k-237-betae/FB15k_mid2name.txt", "r", encoding='utf-8') as f:
    ent2name = {}
    for line in f:
        mid, name = line.strip().split("\t", 1)  # Use maxsplit=1 in case name has tabs
        ent2name[mid] = name


In [5]:
with open("knowledge_graph/queries/CWQ_sim_queries.pkl", "rb") as f:
    queries = pkl.load(f)

In [6]:
class Config:
    data_path = 'knowledge_graph/KG_data/FB15k-237-betae'
    seed = 1234
    k_rel = 3 # beams
    k_cands = 200
    depth = 2 # max depth of subgraph if BFS
    cands_lim = 1024
    gpu = 0
    fact_ratio = 1.0
    val_num = -1 # how many triples are used as the validate set
    add_manual_edges = False
    remove_1hop_edges = True
    not_shuffle_train = False
    device = "cuda:0"
args = Config()
loader = DataLoader(args, mode='train')

# loader.shuffle_train()
train_graph = loader.train_graph
train_graph_homo = list(set([(h,t) for (h,r,t) in train_graph]))

args.n_ent = loader.n_ent
args.n_rel = loader.n_rel

544230 0
==> removing 1-hop links...
==> done


In [7]:
GoG_args = {
    'drop_ratio': 0.6,
}

sampler = ExpandSubgraph(
    args.n_ent, args.n_rel, train_graph_homo, train_graph, args,
    GoG_simulation=False,
    GoG_args=GoG_args
)


In [8]:
def count_path_to_ans(id_query, subgraph_data):
    sum_cnt = 0
    for ans in queries[id_query]['answers_id']:
        cnt = 0
        for edges in subgraph_data[2]:
            if ans in edges[[0,2]]:
                cnt += 1
        sum_cnt += cnt
        # print(ent2name[id2ent[ans]], "|| count: " + str(cnt))
        return sum_cnt

In [9]:
with open("knowledge_graph/queries/transformed_queries2.pkl", "rb") as f:
    queries = pkl.load(f)
# id_query = 194 + 23 + 23
# len(queries), queries[id_query]
# for i in range(len(queries)):
#     queries[i]['natural_query'] = queries[i]['transformed_query']
# queries[0]


In [10]:
reasoning_queries = [q for q in queries if len(q['answers']) == 1]
print(len(reasoning_queries))

247


In [11]:
pkl.dump(reasoning_queries, open("knowledge_graph/queries/CWQ_sim_queries2.pkl", "wb"))

In [23]:
f = pkl.load(open("data_for_GNN_finetune_another_way.pkl", "rb"))
f[0]['subgraph']

(tensor([    6,    30,    41,  ..., 13704, 13725, 13772]),
 tensor([0, 0, 0,  ..., 0, 0, 0]),
 tensor([[3793,  107, 6381],
         [3793,  155, 3605],
         [3793,  106, 6381],
         ...,
         [3142,  155, 1685],
         [3142,  239, 2016],
         [3142,  155,  803]]))

In [ ]:
# subgraph_edges = torch.tensor(f[15]['subgraph'][2])
# for edges in subgraph_edges:
#     inv_rel_idx = edges[1] + 1 if edges[1] % 2 == 0 else edges[1] - 1
#     inv_edge = torch.tensor([edges[2], inv_rel_idx, edges[0]])
#     # Check if inv_edge in subgraph_edges
#     if not any((subgraph_edges[:,0] == inv_edge[0]) & (subgraph_edges[:,1] == inv_edge[1]) & (subgraph_edges[:,2] == inv_edge[2])):
#         # print(f"Missing inverse edge for {edges}: {inv_edge}")
#         assert False, f"Missing inverse edge for {edges}: {inv_edge}"

C:\Users\MTBH\AppData\Local\Temp\ipykernel_6216\1427328866.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  subgraph_edges = torch.tensor(f[15]['subgraph'][2])


AssertionError: Missing inverse edge for tensor([  727,   272, 12083]): tensor([12083,   273,   727])

In [ ]:
from utils import extract_notations

In [ ]:
cnt = {1:0, 2:0, 3:0, 4:0}
for query in queries:
    notations = extract_notations(query['query_type'])
    # print(notations)
    cnt[len(notations)] += 1
    # break
print(cnt)

{1: 0, 2: 199, 3: 162, 4: 144}


In [ ]:
subgraph_datas=[]
num_data = 0
subgraph_size = []
# random.shuffle(queries)
for i, query in enumerate(queries):
    # print(queries[i])
    if len(query['gold_path']) == 0:
        continue
    # sampler.assign_query(query)
    subgraph_data = sampler.sampleSubgraph()
    drop_edges = sampler.drop_edges
    # inv_r = triplet[1] + 1 if triplet[1] % 2 == 0 else triplet[1] - 1
    # inv_edges = [(triplet[2], triplet[1] + inv_r, triplet[0]) for triplet in drop_edges]
    # for edge in inv_edges:
    #     drop_edges.append(edge)
    # print(drop_edges)
    ## Add inverse edges 
    drop_edges = list(set(drop_edges))
    if len(drop_edges) == 0:
        continue
    num_data += len(drop_edges)
    subgraph_datas.append({
        'id_query': i,
        'subgraph': subgraph_data,
        'drop_edges': drop_edges,
    })
    subgraph_size.append(len(subgraph_data[2])) 
    # break
print("Total links: ", num_data)
print("Total subgraphs: ", len(subgraph_datas))
print("Average subgraph size: ", sum(subgraph_size)/len(subgraph_size))


Total links:  1015582
Total subgraphs:  483
Average subgraph size:  3083.9792960662526


In [ ]:
import copy
subgraph_data_t = copy.deepcopy(subgraph_datas)

In [ ]:
total_drop = 0
for data in subgraph_data_t:
    drop_edges = data['drop_edges']
    np.random.shuffle(drop_edges)
    num_drop = int(len(drop_edges) * 0.02)
    
    drop_edges = drop_edges[:num_drop]
    inv_r = lambda r: r + 1 if r % 2 == 0 else r - 1
    inv_edges = [(edge[2], inv_r(edge[1]), edge[0]) for edge in drop_edges]
    drop_edges.extend(inv_edges)
    data['drop_edges'] = drop_edges
    total_drop += len(data['drop_edges'])
print("Total edges to drop: ", total_drop)

Total edges to drop:  40148


In [ ]:
unique_edges = []
new_data = []
for i, d in enumerate(subgraph_data_t):
    # print(d)
    tmp = torch.Tensor(d['drop_edges']).int()
    try:
        edges = torch.unique(tmp[:,1])
        new_data.append(d)
    except:
        print(i)
        # raise ValueError("Error in unique edges")
    unique_edges.append(edges)
    
t_unique_edges = torch.unique(torch.cat(unique_edges))
print(len(t_unique_edges))
# unique_edges


110
294
458


In [ ]:
for i, data in enumerate(new_data):
    subgraph_edges = data['subgraph'][2]
    drop_edges = data['drop_edges']
    # print("aa")
    # Convert drop_edges to tensors to match subgraph_edges dtype
    drop_edges_tensor = torch.tensor(drop_edges, dtype=subgraph_edges.dtype)
    drop_set = set(tuple(edge.tolist()) for edge in drop_edges_tensor)
    
    # Keep only edges not in drop_set
    mask = torch.tensor([tuple(edge.tolist()) not in drop_set for edge in subgraph_edges])
    # print(mask[:10])
    # print((mask == True).any())
    # if  not mask.sum().item() == len(subgraph_edges) -  len(drop_edges):
        # print(len(subgraph_edges), len(mask))
        # print(mask.sum().item(), len(subgraph_edges) -  len(drop_edges))
        # print("Nope", i)
        # break
    data['subgraph'] = (data['subgraph'][0], data['subgraph'][1], subgraph_edges[mask])

In [ ]:
total_valid_query = 0
total_query = 0
final_data = []
for i, query in enumerate(new_data):
    subgraph_edges = query['subgraph'][2].cpu().numpy()
    subgraph_nodes = query['subgraph'][0].cpu().numpy()
    # print(subgraph_edges.shape)
    drop_edges = query['drop_edges']

    ok = 1
    for e in drop_edges:
        dir_triplet = np.array([e[0], e[1], e[2]])
        assert not np.any(np.all(subgraph_edges == dir_triplet, axis=1)) 
        inv_rel = e[1] + 1 if e[1] % 2 == 0 else e[1] - 1
        inv_triplet = np.array([e[2], inv_rel, e[0]])
        assert not np.any(np.all(subgraph_edges == inv_triplet, axis=1)) 
        # total_valid_query +=  ((e[0] in subgraph_nodes) & (e[2] in subgraph_nodes))
        total_query += 1
        if not ((e[0] in subgraph_nodes) & (e[2] in subgraph_nodes)):
            ok = 0
            # break
    if ok:
        total_valid_query += len(drop_edges)
        final_data.append(query)

print("Total queries: ", total_query)
print("Total valid dropped edges: ", total_valid_query)
print("Valid dropped edges ratio: ", total_valid_query / total_query)
print("Len Subgraph datas: ", len(final_data))

Total queries:  40148
Total valid dropped edges:  37126
Valid dropped edges ratio:  0.9247285045332271
Len Subgraph datas:  460


In [ ]:
total_valid_query = 0
total_query = 0
for i, query in enumerate(final_data):
    subgraph_edges = query['subgraph'][2].cpu().numpy()
    subgraph_nodes = query['subgraph'][0].cpu().numpy()
    # print(subgraph_edges.shape)
    drop_edges = query['drop_edges']
    for e in drop_edges:
        dir_triplet = np.array([e[0], e[1], e[2]])
        assert np.any(np.all(subgraph_edges == dir_triplet, axis=1)) == False
        inv_rel = e[1] + 1 if e[1] % 2 == 0 else e[1] - 1
        inv_triplet = np.array([e[2], inv_rel, e[0]])
        assert np.any(np.all(subgraph_edges == inv_triplet, axis=1)) == False
        total_valid_query += ((e[0] in subgraph_nodes) & (e[2] in subgraph_nodes))
        total_query += 1

print("Total queries: ", total_query)
print("Total valid dropped edges: ", total_valid_query)
print("Valid dropped edges ratio: ", total_valid_query / total_query)

Total queries:  37126
Total valid dropped edges:  37126
Valid dropped edges ratio:  1.0


# Output data

In [ ]:
# with open("data_for_GNN_finetune_another_way.pkl", "wb") as f:
#     pkl.dump(final_data, f)